# Synthetic data generation with LLMs

This Lab session leverages the capabilities of Large Language Models (LLMs) to generate synthetic data. It is built upon the [LangChain](https://python.langchain.com/docs/get_started/introduction) framework.

## Key Features
- **Flexible LLM Integration**: In this case we will use  [Ollama](https://ollama.com/)  following the guidance available on the [LangChain Quickstart Guide](https://python.langchain.com/docs/get_started/quickstart#llm-chain).
- **Customizable Data Generation**: Users can tailor the synthetic data generation process through a set of predefined constants, allowing for the adjustment of shop type, the number of categories, vendors, and products, as well as the retry logic for product detail generation.

## Workflow
- **Film Genre List Generation**: Initiates the process by creating a list of unique film genres to review.
- **Synthetic Film Reviews Generation**: For each film genre, a detailed review is generated, including the reasons to like or dislike a film from such genre.

## Interactivity and Customization
Designed within the interactive environment of a Jupyter Notebook, this tool offers users the flexibility to:

- **Monitor and Adjust**: Follow the generation process step-by-step, making real-time adjustments as needed.
- **Rerun and Refine**: Easily rerun sections to refine the outcomes or explore different configurations.
- **Manual Overrides**: Directly edit the generated data, such as vendor names or category specifics, to align with specific preferences or requirements.
This approach not only automates the creation of synthetic data but also places control in the hands of users, blending automation with customization to meet diverse needs.


## Environment Setup

First of all, let's define the LD_LIBRARY_PATH to the CUDA installation. This is required to use Ollama with gpu support.

In [ ]:
import os

# Set LD_LIBRARY_PATH so the system NVIDIA library becomes preferred
# over the built-in library. This is particularly important for
# Google Colab which installs older drivers
os.environ.update({'LD_LIBRARY_PATH': '/usr/lib64-nvidia'})
# Allow Ollama to handle multiple concurrent requests (must be set before server starts)
os.environ['OLLAMA_NUM_PARALLEL'] = '4'

Now, let's install ollama, langchain and langchain's ollama wrapper.

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
!pip install langchain langchain-core langchain-ollama pydantic>=2

We now import the modules that we will use for this lab.

In [ ]:
# Standard library imports
from typing import Dict, List, Optional
import logging
import random

# Related third-party imports
import pandas as pd
from tqdm import tqdm

# Local application/library specific imports
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.output_parsers.list import CommaSeparatedListOutputParser
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

## Ollama

Ollama is a library that provides methods to use LLMs efficiently, as well as integration with langchain.

**Exercise 1**: Read carefully these steps to understand how the model is being hosted in this Colab Session.

**Step 1:** We launch an ollama server on the background. We use `nohup` + `&` to run the thread in the background and not be killed when the notebook cell finishes

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &

**Step 2:** We pull the LLM that we want (Gemma 3 4B) to run on the server. Gemma 3 is a lightweight model (~3GB VRAM) that excels at instruction following and structured output. Check [here](https://ollama.com/library/gemma3) for other available sizes.

In [ ]:
!ollama pull gemma3:4b

If everything is working properly, you should see your Gemma 3 instance by running the following command:

In [ ]:
!ollama list

**Step 3:** Now, let's import the python wrapper that allows us to connect an ollama server with langchain

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:4b", temperature=0.7)

We can now call the model using the `invoke()` method

In [ ]:
response = llm.invoke("Who invented the telescope?")
print(response.content)

By calling ChatOllama we create the python object that will call the Ollama server we created using the terminal. Note that ChatOllama returns an `AIMessage` object, so we access `.content` to get the text.

## Creating our data generation pipeline

Once we have everything ready to use our LLM, let's think about how to generate our synthetic data.

First, let's define some parameters for our generation process:



*   **NUMBER_OF_REVIEWS:** How many synthetic examples we want to create for our tasks. More examples will allow us to train more powerful models, but they will take some time.
*   **NUMBER_OF_GENRES:** Number of film genres to write reviews about. More genres will lead to a more diverse set of reviews.
*  **MAX_ATTEMPS:** To ensure a consistent format for our reviews, we will request our models to generate json files. Sometimes, especially when using small models, the outputs may contain format errors that cannot be properly parsed as jsons. This parameters the maximum number of generation attents before descarding a given topic and moving to a new generation.



In [ ]:

# Number of reviews to generate
NUMBER_OF_REVIEWS = 1000 # Set a lower number for experimentation (e.g. 250), when everything works, generate more samples

#Number of movie genres
NUMBER_OF_GENRES = 15

# Maximum number of attempts for the LLM to generate each product
MAX_ATTEMPTS = 5

## Creation of movie genres
As you know, when using an LLM we usually define a system prompt that sets the main guidelines on how the model should behave.

**Exercise 2**: Write a system prompt to follow the instructions given by the user. The main aspects that you may include in your prompt are:


*  "Who is" the model
*  The model must follow the instructions provided.  
*  The model must not add any additional information.




In [ ]:
# TODO: Prepare the helper prompt
helper_system_prompt = ("...")
                
csl_parser = CommaSeparatedListOutputParser()

prompt_template = PromptTemplate(
    template="{system_prompt}\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={
        "format_instructions": csl_parser.get_format_instructions(),
        "system_prompt": system_prompt,
    },
)

Using this system prompt, we will now define the pipeline used to generate our synthetic data. We will follow these three steps:



1.   **Prompt setup**: Generate a user prompt so that the LLM performs our desired task ("query" in the `prompt_template`).
2.   **LLM inference**: Run the model with the defined input.
3.   **Comma Separated List parser**: Parse the generated text as a python list.



In [ ]:
inference_pipe = prompt_template | llm | csl_parser

**Exercise 3:** Write a prompt asking the model to create a list of `NUMBER_OF_GENRES` movie genres. The main aspects that you should include are:


*   List popular film genres.
*   Try to cover a wide range of categories.



In [ ]:
# TODO: Prepare the genres prompt
genres_prompt = (f"...")
                    

Check that the list make sense and that genres are properly splitted in a list

In [ ]:
genres

## Creation of movie reviews
We define a new class (`Review`) that will define the format in which we expect the output of the model. Then, we create a `JsonOutputParser` object with the same structure as the `Review` class. The rest is almost the same as in the previous case.

To counteract potential failures of the model, especially with smaller LLMs that might not adhere strictly to instructions, we define a retry mechanism. This allows the prompt to be executed up to ```MAX_ATTEMPTS``` times, ensuring the generation of accurate and complete information with the correct format.

We will reuse the system prompt we defined before, and just create a new user prompt that defines the new task.



In [ ]:
class Review(BaseModel):
    title: str = Field(description="Movie title")
    sentiment: str = Field(description="Sentiment (Positive or Negative)")
    genre: str = Field(description="Movie genre")
    review: str = Field(description="Movie review")

json_parser = JsonOutputParser(pydantic_object=Review)

product_prompt = PromptTemplate(
    template="{system_prompt}\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={
        "format_instructions": json_parser.get_format_instructions(),
        "system_prompt": system_prompt,
    },
)

product_chain = product_prompt | llm | json_parser

**Exercise 4:** Write a prompt to generate a movie review given a specific genre and sentiment. The review must clearly explain a negative or positive opinion about the film. The main aspects to include:


*   The review and title must follow the genre provided.
*   Reviews must be brief.
*   Include relevant details about the film, if necessary.

Sometimes it is useful to be repetitive with which is the expected output.


In [ ]:
def generate_product_details(sentiment, genre, chain):
    # TODO: Create the prompt for generating product details
    prompt = (f"...")
                                

Test that the prompt works:

In [ ]:
test_reviews = []
for i in range(3):
    genre = random.choice(genres)
    sentiment = random.choice(['positive', 'negative'])
    review = generate_product_details(sentiment, genre, product_chain)
    if review:
        test_reviews.append(review)
        print(f"[{i+1}/3] {review}")
    else:
        print(f"[{i+1}/3] FAILED for {sentiment} {genre}\n")

print(f"Quick test: {len(test_reviews)}/3 succeeded")

## Generating the synthetic data
Finally, our synthetic reviews are generated. This process may take a while (you may optimize it by increasing `BATCH_CONCURRENCY`). You can also reduce the number of reviews, but it will affect the performance of the final classifier.

In [ ]:
import random
from concurrent.futures import ThreadPoolExecutor, as_completed

BATCH_CONCURRENCY = 8

inputs = []
for _ in range(NUMBER_OF_REVIEWS):
    genre = random.choice(genres)
    sentiment = random.choice(['positive', 'negative'])
    inputs.append((sentiment, genre))

reviews = []
with ThreadPoolExecutor(max_workers=BATCH_CONCURRENCY) as executor:
    futures = {
        executor.submit(generate_product_details, sentiment, genre, product_chain): (sentiment, genre)
        for sentiment, genre in inputs
    }
    for future in tqdm(as_completed(futures), total=NUMBER_OF_REVIEWS, desc='Generating Reviews'):
        result = future.result()
        if result is not None:
            reviews.append(result)

print(f"\nSuccessfully generated {len(reviews)} / {NUMBER_OF_REVIEWS} reviews")

In [ ]:
reviews

Check that the reviews make sense and are consistent with the requested sentiment.

## Saving our synthetic dataset.
We will now save the data (just in case), so that we don't need to generate it again. The data is saved to a pandas DataFrame.


In [ ]:
# Convert the list of dictionaries into a DataFrame
df = pd.DataFrame(reviews)

# Display the first few rows of the DataFrame to verify
print(df.head())

In [ ]:
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # You can modify this path
    path = '/content/drive/MyDrive/synthetic_reviews.pkl'
except ImportError:
    path = 'synthetic_reviews.pkl'
    print("Not running in Google Colab, saving locally.")

In [ ]:
df.to_pickle(path)

print(f"Reviews saved to {path}")

In [ ]:
# In case you need to read it
# df = pd.read_pickle(path)

# print(f"Reviews loaded from {path}")

You may also want to save the data as a CSV

In [ ]:
df.to_csv('reviews.tsv', sep='\t', index=False)

## Train a sentiment analysis model


Finally, we want to see if our synthetic data is useful for the task we want to solve. Following the methods we saw on the NLP module, we will create a small neural network classifier to test it.

First, we split our data into train/test/dev splits removing generation errors.

In [ ]:
import random
import itertools
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split

SEED = 0
train_sents = df['review'].astype(str).apply(str.split).tolist()
train_targets = [1 if str(s).strip().lower() == 'positive' else 0 for s in df['sentiment']]

train_sents, test_sents, train_targets, test_targets = train_test_split(train_sents,
                                                                          train_targets,
                                                                          test_size=0.20,
                                                                          random_state=SEED)
valid_sents = test_sents[:100]
valid_targets = test_targets[:100]

test_sents = test_sents[100:]
test_targets = test_targets[100:]


And learn a vocabulary from the training split.

In [ ]:
def compute_vocabulary(train_sents):
  vocabulary = {'<unk>':99999999,'<pad>':99999999}
  for sent in train_sents:
    for token in sent:
      if token in vocabulary:
        vocabulary[token] += 1
      else:
        vocabulary[token] = 1

  return vocabulary

voc = compute_vocabulary(train_sents)

**Exercise 6:** Create a neural network classifier to test our generated data. As the dataset is small, we will use a simple network using only the following layers:

1. Embedding layer: This layer computes a continuous vector representation for each one of the tokens of the vocabulary.
2. Pooling layer: This layer picks the maximum value for each dimension of the embedding for all words in the sentence. As a result the whole sentence is represented as a single vector of size embedding_dim.
3. Linear layer that transform the output of the previous layer into the number number of outputs of the network. In this case just one.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Network(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc = nn.Linear(embedding_dim, output_dim)

    def forward(self, x):

        # TODO
        pooled = ...

        #pooled = [batch size, embedding_dim]

        # TODO
        return ...

    def encode(self, x):

        #x = [sent len, batch size]
        # TODO
        embedded = ...

        #embedded = [sent len, batch size, emb dim]

        # TODO
        embedded =

        #embedded = [batch size, sent len, emb dim]

        # TODO
        pooled = F.avg_pool2d(...).squeeze(1)

        return pooled

We define some hyperparameters for our network

In [ ]:
INPUT_DIM = len(voc)
EMBEDDING_DIM = 128
OUTPUT_DIM = 1
BATCH_SIZE = 32

model = Network(INPUT_DIM, EMBEDDING_DIM, OUTPUT_DIM)

In [ ]:
import numpy as np

def idx_voc(voc):
  items = list(voc.items())
  items.sort(key=lambda x: x[1], reverse=True)
  return {item[0]:n for n,item in enumerate(items)}

def token2idx(sents, voc, pad_len=None):
  data = []
  ivoc = idx_voc(voc)

  for sent in sents:
      data.append([ivoc[t] if t in voc else ivoc['<unk>'] for t in sent])

  if pad_len == None:
    pad_len = max([len(sent) for sent in data])

  pad_data = np.full((len(data),pad_len),ivoc['<pad>'])

  for i,sent in enumerate(data):
    trunc_len = min(len(sent), pad_len)
    pad_data[i][:trunc_len] = sent[:trunc_len]

  return pad_data,pad_len


And prepare our data to be used as input for the model. Remember that in order to train using batches we must add padding to ensure that all sentences in the batch have the same number of tokens.

In [ ]:
pad_len = 100

train_idx, _ = token2idx(train_sents, voc, pad_len)
train_targets = np.array(train_targets, dtype='int64')

valid_idx, _ = token2idx(valid_sents, voc, pad_len)
valid_targets = np.array(valid_targets).astype('int64')

test_idx, _ = token2idx(test_sents, voc, pad_len)
test_targets = np.array(test_targets).astype('int64')

And setup all the necessary functions, optimizer and loss function for our model.

In [ ]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr=1e-3)

criterion = nn.BCEWithLogitsLoss()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = model.to(device)
criterion = criterion.to(device)

In [ ]:
def binary_accuracy(preds, y):
    """
    Returns accuracy per batch, i.e. if you get 8/10 right, this returns 0.8, NOT 8
    """

    #round predictions to the closest integer
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float() #convert into float for division
    acc = correct.sum()/len(correct)
    return acc

In [ ]:
def train(model, X, Y, optimizer, criterion, batch_size):

    epoch_loss = 0
    epoch_acc = 0
    n_batch = 0

    model.train()

    permutation = torch.randperm(X.size()[0])

    for i in range(0, X.size()[0], batch_size):

        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X[indices], Y[indices]
        optimizer.zero_grad()
        predictions = model(batch_x).squeeze(1)
        loss = criterion(predictions, batch_y)
        acc = binary_accuracy(predictions, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()
        n_batch += 1

    return epoch_loss / max(n_batch, 1), epoch_acc / max(n_batch, 1)

In [ ]:
def evaluate(model, X, Y, criterion, batch_size):

    epoch_loss = 0
    epoch_acc = 0
    n_batch = 0

    model.eval()

    with torch.no_grad():

        permutation = torch.randperm(X.size()[0])

        for i in range(0, X.size()[0], batch_size):

          indices = permutation[i:i+batch_size]
          batch_x, batch_y = X[indices], Y[indices]

          predictions = model(batch_x).squeeze(1)
          loss = criterion(predictions, batch_y)
          acc = binary_accuracy(predictions, batch_y)

          epoch_loss += loss.item()
          epoch_acc += acc.item()
          n_batch += 1

    return epoch_loss / max(n_batch, 1), epoch_acc / max(n_batch, 1)

We train our network and evaluate it on the validation set. You can set the number of epochs to ensure that the model is not overfitting.

In [ ]:
N_EPOCHS = 10

train_x = torch.LongTensor(train_idx).to(device)
train_y = torch.Tensor(train_targets).to(device)

valid_x = torch.LongTensor(valid_idx).to(device)
valid_y = torch.Tensor(valid_targets).to(device)

for epoch in range(N_EPOCHS):
    train_loss, train_acc = train(model, train_x, train_y, optimizer, criterion, BATCH_SIZE)
    valid_loss, valid_acc = evaluate(model, valid_x, valid_y, criterion, BATCH_SIZE)
    print(f'| Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}% | Val. Loss: {valid_loss:.3f} | Val. Acc: {valid_acc*100:.2f}% |')

And evaluate it on the test split

In [ ]:
test_x = torch.LongTensor(test_idx).to(device)
test_y = torch.Tensor(test_targets).to(device)

test_loss, test_acc = evaluate(model, test_x, test_y, criterion, BATCH_SIZE)

print(f'| Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}% |')

## Conclusions

In this notebook we have seen a pipeline for synthetic data generation using popular libraries (Ollama and LangChain). We can see that we can achieve good results from synthetic data, but we should keep in mind that their quality greatly depends on the clarity of our prompts.